# Adım 2: Hafta 01 analizi

Bu bölüm yalnızca **tam veri** (2023-2026) çekildikten sonra çalıştırılmalıdır (01_veri_cekme.ipynb, `DENEME_MODU = False`).
Altı grafik ve özet rakamları `03_grafikler/hafta01/` klasörüne kaydeder:

1. Aylık profil (mevsimsellik)
2. Saat x ay ısı haritası
3. Akşam / gündüz fiyat oranı
4. 100 TL/MWh altı saat sayısı
5. Öğle saatlerinde güneş + rüzgâr payı ile öğle fiyatı
6. 2026 ile önceki yılların karşılaştırması

Not: 2026'da Kasım-Aralık olmadığı için her yılın ortalaması yalnızca Ocak-Ekim aylarıyla hesaplanır; böylece yıllar adil karşılaştırılır.


In [ ]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib, logging
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

INK="#111827"; INK2="#4b5563"; MUTE="#9ca3af"; GRID="#e5e7eb"
NAVY="#1f3a5f"; ACC="#d9480f"; YEAR_GRAYS=["#c9ced6","#9aa3af","#5f6b7a"]
plt.rcParams.update({"font.family":["Inter","Arial","DejaVu Sans"],"text.color":INK,"axes.labelcolor":INK2,
    "xtick.color":INK2,"ytick.color":INK2,"axes.edgecolor":"#cfd4da"})
AY=["Oca","Şub","Mar","Nis","May","Haz","Tem","Ağu","Eyl","Eki","Kas","Ara"]
GUNDUZ=range(6,17); AKSAM=range(17,22)   # EPİAŞ gün bölümleri: 06-16, 17-21
DUSUK_ESIK=100   # TL/MWh

def fig(title, sub, size=(12,6.75)):
    f=plt.figure(figsize=size,facecolor="white",dpi=160)
    f.text(0.06,0.93,title,fontsize=20,fontweight="bold",ha="left")
    if sub: f.text(0.06,0.875,sub,fontsize=12.5,color=INK2,ha="left")
    return f
def foot(f, txt):
    f.text(0.06,0.03,txt,fontsize=9.5,color=MUTE)
def clean(ax):
    for s in ("top","right"): ax.spines[s].set_visible(False)
    ax.tick_params(length=0); ax.grid(axis="y",color=GRID,lw=1); ax.set_axisbelow(True)
def year_color(y, last):
    return ACC if y==last else YEAR_GRAYS[max(0,len(YEAR_GRAYS)-(last-y))]
KAYNAK="Kaynak: EPİAŞ Şeffaflık Platformu, saatlik veri."

def load(path):
    df=pd.read_csv(path,parse_dates=["zaman"]).set_index("zaman").sort_index()
    df=df[~df.index.duplicated()]
    df["yil"]=df.index.year; df["ay"]=df.index.month; df["saat"]=df.index.hour
    return df

def run(path, out):
    out=Path(out); out.mkdir(parents=True, exist_ok=True)
    df=load(path); years=sorted(df.yil.unique()); last=years[-1]
    R={}  # özet rakamlar
    # yıllık ortalamaya oran (nominal etkiyi ve tavan değişimini nötrler)
    # Son yıl eksikse (örn. Kasım-Aralık yok) karşılaştırılabilirlik için tüm yıllarda
    # yalnızca son yılda bulunan aylarla yıl ortalaması hesaplanır.
    ref_aylar=sorted(df[df.yil==last].ay.unique()); R["referans_aylar"]=[int(x) for x in ref_aylar]
    ref=df[df.ay.isin(ref_aylar)].groupby("yil").ptf.mean()
    yil_ort=df.yil.map(ref)
    df["oran"]=df.ptf/yil_ort

    # --- G1: aylık profil
    m=df.groupby(["yil","ay"]).ptf.mean().unstack(0)
    mo=m/ m.loc[ref_aylar].mean()
    f=fig("Elektrik fiyatının mevsimi var mı?",f"Ayın ortalama PTF'si / o yılın {AY[ref_aylar[0]-1]}-{AY[ref_aylar[-1]-1]} ortalaması (1,0 = ortalama)")
    ax=f.add_axes([0.07,0.15,0.86,0.66]); clean(ax)
    for y in years:
        s=mo[y].dropna(); c=year_color(y,last); lw=2.6 if y==last else 1.8
        ax.plot(s.index,s.values,color=c,lw=lw,marker="o",ms=4.5 if y==last else 3.5)
        ax.text(s.index[-1]+0.15,s.values[-1],str(y),color=c if y==last else INK2,fontsize=11,
                fontweight="bold" if y==last else "normal",va="center")
    ax.axhline(1,color=INK2,lw=1,ls=(0,(3,3)))
    ax.set_xticks(range(1,13)); ax.set_xticklabels(AY); ax.set_xlim(0.6,12.9)
    foot(f,KAYNAK+" Oranlar nominal TL fiyatlar üzerinden yıl içinde hesaplanmıştır.")
    f.savefig(out/"g1_aylik_profil.png"); plt.close(f)
    R["aylik_oran_ort"]=mo.mean(axis=1).round(3).to_dict()
    R["en_ucuz_ay"]=int(mo.mean(axis=1).idxmin()); R["en_pahali_ay"]=int(mo.mean(axis=1).idxmax())

    # --- G2: saat x ay ısı haritası (tüm yılların ortalaması, yıl ortalamasına oran)
    h=df.groupby(["saat","ay"]).oran.mean().unstack()
    f=fig("Günün hangi saati, hangi ayda pahalı?","Saatlik PTF, yıl ortalamasına oran; 2023-%d ortalaması"%last)
    ax=f.add_axes([0.07,0.13,0.80,0.69])
    cmap=LinearSegmentedColormap.from_list("s",["#f3f6fa","#9fb6d1","#3e6591",NAVY])
    im=ax.imshow(h.values,aspect="auto",cmap=cmap,origin="upper")
    ax.set_xticks(range(12)); ax.set_xticklabels(AY); ax.set_yticks(range(0,24,2))
    ax.set_yticklabels([f"{s:02d}:00" for s in range(0,24,2)])
    for s in ax.spines.values(): s.set_visible(False)
    ax.tick_params(length=0)
    cax=f.add_axes([0.89,0.13,0.015,0.69]); cb=f.colorbar(im,cax=cax); cb.outline.set_visible(False)
    cb.set_label("yıl ortalamasına oran",color=INK2)
    foot(f,KAYNAK+" Koyu renk daha pahalı saatleri gösterir.")
    f.savefig(out/"g2_saat_ay_isi_haritasi.png"); plt.close(f)
    R["en_ucuz_hucre"]=dict(zip(["saat","ay"],[int(x) for x in np.unravel_index(np.nanargmin(h.values),h.shape)]))
    R["en_ucuz_hucre"]["ay"]+=1; R["en_ucuz_hucre"]["oran"]=round(float(np.nanmin(h.values)),3)

    # --- G3: akşam / gündüz oranı
    g=df[df.saat.isin(GUNDUZ)].groupby(["yil","ay"]).ptf.mean()
    a=df[df.saat.isin(AKSAM)].groupby(["yil","ay"]).ptf.mean()
    r=(a/g).unstack(0)
    f=fig("Akşam elektriği gündüzden kaç kat pahalı?","Akşam (17-21) ortalama PTF'nin gündüz (06-16) ortalamasına oranı")
    ax=f.add_axes([0.07,0.15,0.86,0.66]); clean(ax)
    for y in years:
        s=r[y].dropna(); c=year_color(y,last)
        ax.plot(s.index,s.values,color=c,lw=2.6 if y==last else 1.8,marker="o",ms=4)
        ax.text(s.index[-1]+0.15,s.values[-1],str(y),color=c if y==last else INK2,fontsize=11,va="center",
                fontweight="bold" if y==last else "normal")
    ax.axhline(1,color=INK2,lw=1,ls=(0,(3,3)))
    ax.set_xticks(range(1,13)); ax.set_xticklabels(AY); ax.set_xlim(0.6,12.9)
    ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v,_:f"{v:.1f}x".replace(".",",")))
    foot(f,KAYNAK)
    f.savefig(out/"g3_aksam_gunduz_orani.png"); plt.close(f)
    R["aksam_gunduz_yillik"]=(a.groupby(level=0).mean()/g.groupby(level=0).mean()).round(2).to_dict()
    R["aksam_gunduz_max"]={"oran":round(float(r.max().max()),2),
        "yil":int(r.max().idxmax()),"ay":int(r[r.max().idxmax()].idxmax())}

    # --- G4: düşük fiyatlı saat sayısı (yıl x ay tablo)
    low=(df.ptf<DUSUK_ESIK).groupby([df.yil,df.ay]).sum().unstack().reindex(columns=range(1,13))
    f=fig("Çok ucuz saatler hangi aylarda toplanıyor?",f"PTF'nin {DUSUK_ESIK} TL/MWh altında kaldığı saat sayısı")
    ax=f.add_axes([0.10,0.20,0.84,0.58])
    cmap2=LinearSegmentedColormap.from_list("o",["#fbfbfb","#f7c8ad","#ec7a45",ACC])
    ax.imshow(low.values,aspect="auto",cmap=cmap2)
    for i in range(low.shape[0]):
        for j in range(12):
            v=low.values[i,j]
            if not np.isnan(v):
                ax.text(j,i,f"{int(v)}",ha="center",va="center",fontsize=11,
                        color="white" if v>np.nanmax(low.values)*0.55 else INK)
    ax.set_xticks(range(12)); ax.set_xticklabels(AY); ax.set_yticks(range(len(low.index)))
    ax.set_yticklabels(low.index); ax.tick_params(length=0)
    for s in ax.spines.values(): s.set_visible(False)
    foot(f,KAYNAK+f" Eşik: {DUSUK_ESIK} TL/MWh (nominal).")
    f.savefig(out/"g4_dusuk_fiyat_saatleri.png"); plt.close(f)
    R["dusuk_saat_yillik"]=low.sum(axis=1).astype(int).to_dict()

    # --- G5: neden? öğle saatlerinde güneş+rüzgâr payı vs gündüz fiyat oranı (aylık)
    ogle=df[df.saat.isin(range(10,16))]
    ren_cols=[c for c in ["uretim_solar","uretim_wind"] if c in df.columns]
    if ren_cols and "uretim_total" in df.columns:
        pay=(ogle[ren_cols].sum(axis=1).groupby([ogle.yil,ogle.ay]).sum()/
             ogle.uretim_total.groupby([ogle.yil,ogle.ay]).sum())*100
        ogle_oran=(ogle.ptf.groupby([ogle.yil,ogle.ay]).mean()/df.ptf.groupby([df.yil,df.ay]).mean())
        d=pd.DataFrame({"pay":pay,"oran":ogle_oran}).dropna()
        f=fig("Güneş ve rüzgâr payı arttıkça öğle fiyatı ne oluyor?",
              "Her nokta bir ay: öğle (10-15) saatlerinde güneş+rüzgâr payı ve öğle fiyatının ay ortalamasına oranı")
        ax=f.add_axes([0.08,0.15,0.86,0.66]); clean(ax); ax.grid(axis="x",color=GRID)
        for y in years:
            s=d.loc[y] if y in d.index.get_level_values(0) else None
            if s is None: continue
            ax.scatter(s.pay,s.oran,s=60 if y==last else 46,color=year_color(y,last),
                       edgecolor="white",linewidth=1.2,label=str(y),zorder=3)
        b=np.polyfit(d.pay,d.oran,1); xs=np.linspace(d.pay.min(),d.pay.max(),50)
        ax.plot(xs,np.polyval(b,xs),color=INK,lw=1.4,ls=(0,(4,3)))
        ax.legend(frameon=False,loc="upper right",ncol=len(years),fontsize=11)
        ax.set_xlabel("Öğle saatlerinde güneş + rüzgâr payı (%)"); ax.set_ylabel("Öğle PTF / ay ortalaması")
        foot(f,KAYNAK+" Kesikli çizgi doğrusal eğilimdir; ilişki nedensellik iddiası taşımaz, talep ve hidroloji de etkilidir.")
        f.savefig(out/"g5_yenilenebilir_ogle_fiyati.png"); plt.close(f)
        R["ren_ogle_egim_10puan"]=round(float(b[0]*10),3)
        R["ren_ogle_korelasyon"]=round(float(d.pay.corr(d.oran)),3)
        R["ren_ogle_pay_yillik"]=d.pay.groupby(level=0).mean().round(1).to_dict()

    # --- G6: son yıl vs önceki yıllar bandı
    prev=[y for y in years if y<last]
    if prev:
        band=mo[prev]
        f=fig(f"{last}, önceki yılların deseninden nerede ayrılıyor?",
              f"Aylık PTF / yıl ortalaması; gri bant {prev[0]}-{prev[-1]} aralığı")
        ax=f.add_axes([0.07,0.15,0.86,0.66]); clean(ax)
        ax.fill_between(band.index,band.min(axis=1),band.max(axis=1),color="#e3e7ec",lw=0)
        ax.plot(band.index,band.mean(axis=1),color=MUTE,lw=1.6,ls=(0,(3,3)))
        s=mo[last].dropna(); ax.plot(s.index,s.values,color=ACC,lw=2.8,marker="o",ms=5)
        ax.text(s.index[-1]+0.15,s.values[-1],str(last),color=ACC,fontsize=11,fontweight="bold",va="center")
        ax.text(12.05,band.mean(axis=1).iloc[-1],f"{prev[0]}-{prev[-1]}\nortalaması",color=INK2,fontsize=10,va="center")
        ax.set_xticks(range(1,13)); ax.set_xticklabels(AY); ax.set_xlim(0.6,13.3)
        foot(f,KAYNAK)
        f.savefig(out/"g6_son_yil_sapma.png"); plt.close(f)
        sap=(mo[last]-band.mean(axis=1)).dropna()
        R["son_yil_sapma"]={"en_yukari_ay":int(sap.idxmax()),"deger":round(float(sap.max()),3),
                            "en_asagi_ay":int(sap.idxmin()),"deger2":round(float(sap.min()),3)}

    R["yillik_ort_ptf"]=df.groupby("yil").ptf.mean().round(0).to_dict()
    R["kapsam"]=[str(df.index.min()),str(df.index.max()),int(len(df))]
    (out/"ozet.json").write_text(json.dumps(R,ensure_ascii=False,indent=2,default=str),encoding="utf-8")
    return R



In [ ]:
# Analizi çalıştır
from IPython.display import Image as IPImage, display as ipdisplay
VERI  = Path("02_islenmis/ana_tablo_saatlik.csv")
CIKTI = Path("03_grafikler/hafta01")
assert VERI.exists(), "Önce 01_veri_cekme.ipynb ile tam veriyi çekin."
OZET = run(VERI, CIKTI)
for g in sorted(CIKTI.glob("g*.png")):
    print(g.name); ipdisplay(IPImage(filename=str(g), width=900))
print(json.dumps(OZET, ensure_ascii=False, indent=1, default=str))

## Son adım
Şu dosyaları Claude'a yükleyin:
- `02_islenmis/ana_tablo_saatlik.csv`
- `03_grafikler/hafta01/ozet.json`

Claude bunlarla bulguları doğrulayıp 10 sayfalık PDF'i ve gönderi metnini hazırlar.